In [1]:
import sys
import pandas as pd
import numpy as np
import pickle
from pathlib import Path
from scipy.spatial import cKDTree
sys.path.append('/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM')
from src.inference import get_broom

In [2]:
def extract_brooms(protein,
                   start_brooms,
                   max_neighbours):
    
    tree = cKDTree(np.asarray(protein[["x","y","z"]]))
    _distances, neighbors = tree.query(start_brooms, k=max_neighbours)
    pos_list  = []
    cls_list  = []

    for c, nbr in zip(start_brooms, neighbors):
        p, cl = get_broom(c, protein, nbr)
        #print(p.shape)
        pos_list.append(p)
        cls_list.append(cl)

    return (np.concatenate(pos_list, axis = 0),
            np.concatenate(cls_list, axis = 0))


def split_even_two(counts, rng=None):
    counts = np.asarray(counts, dtype=int)
    rng = rng or np.random.default_rng()

    base = counts // 2                      # each side gets at least this
    give_left = rng.integers(0, 2, size=counts.size).astype(bool)
    extra_left = ((counts & 1).astype(bool) & give_left).astype(int)

    left = base + extra_left
    right = counts - left
    return list(zip(left, right))

def draw_uniform(sample_size, probs):
    rng = np.random.default_rng()
    return rng.multinomial(sample_size, np.full(probs, 1.0/probs))


def random_neg_brooms(data_path, n_samples, max_neighbours):

    broom_pos = list()
    broom_cls = list()

    files = list(data_path.glob("*"))
    counts = draw_uniform(n_samples, probs = len(files))

    for path, counts in zip(files,
                            split_even_two(counts)):

        for type_path, count in zip([path / "AB", path / "AG"], counts):

            with open(type_path / "data_target.pkl", "rb") as f:
                protein_target = pickle.load(f)

            random_pts = np.load(type_path / "random_pts.npy")
            idx = np.random.choice(random_pts.shape[0], size=count, replace=False)
            selected_pts = random_pts[idx]

            pos,cls = extract_brooms(protein = protein_target,
                                 start_brooms=selected_pts,
                                 max_neighbours=max_neighbours)

            broom_pos.append(pos)
            broom_cls.append(cls)

    broom_pos = np.concatenate(broom_pos, axis = 0)
    broom_cls = np.concatenate(broom_cls, axis = 0)
    return broom_pos, broom_cls

In [3]:
BASE = Path("/home/kiemax/Helmholtz/projects/proteinTPM/noripTPM/data/sabdab/negativeSamples/randomSampling")
N_SAMPLES = 10000
params = {"MAX_NEIGHBOURS": 25}

In [4]:
pos, cls = random_neg_brooms(data_path = BASE, 
                             n_samples = N_SAMPLES,
                             max_neighbours = params["MAX_NEIGHBOURS"])

In [5]:
pos


array([[[0.6829999999999963, 1.5669999999999993, -3.5740000000000016],
        [0.5039999999999978, 1.7899999999999996, -4.776],
        [-1.0210000000000043, -0.08200000000000063, -5.469000000000001],
        ...,
        [1.269999999999996, 6.129, -3.780999999999999],
        [3.253999999999998, 6.555999999999999, 0.38899999999999935],
        [-0.5480000000000018, 7.369, 0.3979999999999997]],

       [[-2.7790000000000035, 1.6309999999999967, -0.2400000000000002],
        [-2.5660000000000025, 0.5309999999999988, -2.385],
        [2.554999999999996, 1.6889999999999965, -2.103999999999999],
        ...,
        [2.546999999999997, 4.139999999999997, -3.4780000000000015],
        [3.666999999999998, 2.0609999999999964, 4.35],
        [2.1869999999999976, 5.205999999999996, -2.449]],

       [[-1.6810000000000045, -1.2240000000000002, 2.0420000000000016],
        [-1.461000000000002, -0.23900000000000077, 2.7830000000000013],
        [-0.4070000000000036, -0.1070000000000002, 3.4410000